In [1]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

/kaggle/input/competitions/ieee-fraud-detection/sample_submission.csv
/kaggle/input/competitions/ieee-fraud-detection/test_identity.csv
/kaggle/input/competitions/ieee-fraud-detection/train_identity.csv
/kaggle/input/competitions/ieee-fraud-detection/test_transaction.csv
/kaggle/input/competitions/ieee-fraud-detection/train_transaction.csv


In [3]:
!pip install dagshub mlflow
import pandas as pd
import mlflow
import dagshub
import gc

dagshub_username = "nikaduri"
repo_name = "ml-ieee-cis-fraud-detection"
dagshub.init(repo_owner=dagshub_username, repo_name=repo_name, mlflow=True)

❗❗❗ AUTHORIZATION REQUIRED ❗❗❗

Output()



Open the following link in your browser to authorize the client:
https://dagshub.com/login/oauth/authorize?state=60633e74-94df-4b2c-a81e-81b6e02fe291&client_id=32b60ba385aa7cecf24046d8195a71c07dd345d9657977863b52e7748e0f0f28&middleman_request_id=e74321c758d7ef6816aa6c8b9b23ac957101a1f66bceed4340db12e9b3efa4b7




Accessing as nikaduri

Initialized MLflow to track repo "nikaduri/ml-ieee-cis-fraud-detection"

Repository nikaduri/ml-ieee-cis-fraud-detection initialized!

In [8]:
test_transaction = pd.read_csv('/kaggle/input/competitions/ieee-fraud-detection/test_transaction.csv')
test_identity = pd.read_csv('/kaggle/input/competitions/ieee-fraud-detection/test_identity.csv')

test_df = test_transaction.merge(test_identity, on='TransactionID', how='left')

test_df.columns = test_df.columns.str.replace('-', '_')

submission = pd.DataFrame({'TransactionID': test_df['TransactionID']})

del test_transaction, test_identity
gc.collect()

104

In [5]:
model_name = "XGBoost_Production_Model"

model_uri = f"models:/{model_name}/latest"

production_pipeline = mlflow.sklearn.load_model(model_uri)

In [9]:
probabilities = production_pipeline.predict_proba(test_df)[:, 1]

submission['isFraud'] = probabilities


submission_path = 'submission.csv'
submission.to_csv(submission_path, index=False)

submission.head()

🚀 Success! Submission saved to submission.csv.
Total rows predicted: 506691


,TransactionID,isFraud
0,3663549,0.110015
1,3663550,0.261291
2,3663551,0.151676
3,3663552,0.065656
4,3663553,0.225625
